# Negative weights and all pairs - Bellman-Ford, SPFA, Floyd-Warshall

Dijkstra is fast because it *settles* a vertex: once popped, its distance is final.
That is only sound when no edge can shorten a path later, i.e. when every weight is
non-negative. Currencies, chemical potentials, profit-and-loss on a trade and any
"reward" edge break that assumption, and the algorithm quietly returns a wrong number.

Three answers, all older than they look:

| | idea | cost |
|---|---|---|
| **Bellman-Ford** | settle nothing; relax every edge `V-1` times | `O(VE)` |
| **SPFA** | Bellman-Ford, but only re-relax vertices that changed | `O(VE)` worst case |
| **Floyd-Warshall** | DP over "which vertices may appear in the middle", all pairs | `O(V^3)` |

In [1]:
from collections import deque
from itertools import product
INF = float('inf')

# CLRS's classic example: negative edges, but no negative cycle.
NAMES = 'ABCDE'
N = 5
EDGES = [(0, 1, 6), (0, 3, 7),
         (1, 2, 5), (1, 3, 8), (1, 4, -4),
         (2, 1, -2),
         (3, 2, -3), (3, 4, 9),
         (4, 0, 2), (4, 2, 7)]

def build_adj(n, edges):
    adj = [[] for _ in range(n)]
    for u, v, w in edges:
        adj[u].append((v, w))           # directed
    return adj

adj = build_adj(N, EDGES)
for u, v, w in EDGES:
    print('%s -> %s  %3d' % (NAMES[u], NAMES[v], w))

A -> B    6
A -> D    7
B -> C    5
B -> D    8
B -> E   -4
C -> B   -2
D -> C   -3
D -> E    9
E -> A    2
E -> C    7


## Bellman-Ford

One rule, repeated: if `dist[u] + w < dist[v]`, write the smaller value into `dist[v]`.
That is *relaxation*, exactly the same line Dijkstra uses - Bellman-Ford just refuses to
decide in which order to apply it, and instead applies it to everything.

After round `r`, every shortest path that uses at most `r` edges has been found. A
shortest path in a graph without negative cycles never repeats a vertex, so it uses at
most `V-1` edges, and `V-1` rounds are always enough. If a round changes nothing, the
answer has converged and we can stop early.

In [2]:
def bellman_ford(n, edges, src):
    """Returns (dist, parent, negative_cycle_reachable)."""
    dist, parent = [INF] * n, [-1] * n
    dist[src] = 0
    for _ in range(n - 1):
        changed = False
        for u, v, w in edges:
            if dist[u] + w < dist[v]:   # relax
                dist[v] = dist[u] + w
                parent[v] = u
                changed = True
        if not changed:                 # converged early
            break
    for u, v, w in edges:               # one extra round
        if dist[u] + w < dist[v]:       # still improving -> negative cycle
            return dist, parent, True
    return dist, parent, False

def rounds(n, edges, src):
    dist = [INF] * n
    dist[src] = 0
    snaps = [dist[:]]
    for _ in range(n - 1):
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
        snaps.append(dist[:])
    return snaps

fmt = lambda d: '  '.join('%s=%s' % (NAMES[i], '-' if x == INF else x) for i, x in enumerate(d))
for r, d in enumerate(rounds(N, EDGES, 0)):
    print('after round %d :  %s' % (r, fmt(d)))

dist, parent, neg = bellman_ford(N, EDGES, 0)
print('final        :  %s   negative cycle: %s' % (fmt(dist), neg))

after round 0 :  A=0  B=-  C=-  D=-  E=-
after round 1 :  A=0  B=6  C=4  D=7  E=2
after round 2 :  A=0  B=2  C=4  D=7  E=2
after round 3 :  A=0  B=2  C=4  D=7  E=-2
after round 4 :  A=0  B=2  C=4  D=7  E=-2
final        :  A=0  B=2  C=4  D=7  E=-2   negative cycle: False


Notice `B` going `6 -> 2` in round 2: Dijkstra would already have settled `B` at 6 and
never revisited it. The cheaper route `A -> D -> C -> B` costs `7 - 3 - 2 = 2`, and only
becomes visible after the negative edges have been relaxed.

## SPFA - only re-check what changed

Most edges relax into nothing. SPFA (Shortest Path Faster Algorithm) keeps a queue of
vertices whose `dist` actually improved and only scans those. It is Bellman-Ford with a
work list - same answers, same `O(VE)` worst case, usually far fewer relaxations.

It also detects negative cycles for free: a vertex cannot be pushed `V` times unless
something keeps getting cheaper, which only a negative cycle can do.

In [3]:
def spfa(n, adj, src):
    dist = [INF] * n
    dist[src] = 0
    inq, pushes = [False] * n, [0] * n
    q = deque([src]); inq[src] = True
    pops = 0
    while q:
        u = q.popleft(); inq[u] = False; pops += 1
        for v, w in adj[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                if not inq[v]:
                    pushes[v] += 1
                    if pushes[v] >= n:          # negative cycle
                        return dist, pops, True
                    q.append(v); inq[v] = True
    return dist, pops, False

sdist, pops, sneg = spfa(N, adj, 0)
print('dist :', fmt(sdist))
print('queue pops: %d   vs Bellman-Ford scanning %d edges %d times = %d'
      % (pops, len(EDGES), N - 1, len(EDGES) * (N - 1)))

dist : A=0  B=2  C=4  D=7  E=-2
queue pops: 7   vs Bellman-Ford scanning 10 edges 4 times = 40


## When the answer is minus infinity

Change one weight - `E -> C` from 7 to 1 - and the cycle `B -> E -> C -> B` costs
`-4 + 1 - 2 = -5`. Going round it again is always cheaper, so "the shortest path" no
longer exists. Both algorithms report it rather than looping forever.

In [4]:
def find_negative_cycle(n, edges):
    dist, parent = [0] * n, [-1] * n    # 0 everywhere = virtual source into every vertex
    x = -1
    for _ in range(n):
        x = -1
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                parent[v] = u
                x = v
    if x == -1:
        return None
    for _ in range(n):                  # walk back until we are inside the cycle
        x = parent[x]
    cycle, v = [], x
    while True:
        cycle.append(v)
        v = parent[v]
        if v == x:
            break
    cycle.append(x); cycle.reverse()
    return cycle

neg_edges = [(u, v, 1 if (u, v) == (4, 2) else w) for u, v, w in EDGES]
_, _, has_neg = bellman_ford(N, neg_edges, 0)
_, _, spfa_neg = spfa(N, build_adj(N, neg_edges), 0)
cyc = find_negative_cycle(N, neg_edges)
print('Bellman-Ford (extra round still improves):', has_neg)
print('SPFA (a vertex queued n times)          :', spfa_neg)
print('cycle:', ' -> '.join(NAMES[v] for v in cyc))

Bellman-Ford (extra round still improves): True
SPFA (a vertex queued n times)          : True
cycle: B -> E -> C -> B


## Floyd-Warshall - all pairs, three loops

`dist[i][j]` after stage `k` means: the cheapest `i -> j` path whose **intermediate**
vertices all come from `{0..k}`. Adding vertex `k` gives exactly two choices - route
through `k` or not - so the update is

    dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])

That is the whole algorithm. `k` must be the **outer** loop, because `dist[i][k]` and
`dist[k][j]` have to be final for stage `k-1` before stage `k` uses them.

In [5]:
def floyd_warshall(n, edges):
    dist = [[INF] * n for _ in range(n)]
    nxt = [[-1] * n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0
    for u, v, w in edges:
        if w < dist[u][v]:
            dist[u][v], nxt[u][v] = w, v
    for k in range(n):                  # k MUST be outermost
        for i in range(n):
            if dist[i][k] == INF:
                continue
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
                    nxt[i][j] = nxt[i][k]
    return dist, nxt

def floyd_path(nxt, i, j):
    if nxt[i][j] == -1:
        return []
    path = [i]
    while i != j:
        i = nxt[i][j]; path.append(i)
    return path

fdist, nxt = floyd_warshall(N, EDGES)
print('     ' + '  '.join('%4s' % c for c in NAMES))
for i in range(N):
    print('%s   ' % NAMES[i] + '  '.join(
        '%4s' % ('-' if fdist[i][j] == INF else fdist[i][j]) for j in range(N)))
print('row A == Bellman-Ford from A:', fdist[0] == dist)
print('C -> D goes', ' -> '.join(NAMES[v] for v in floyd_path(nxt, 2, 3)))

        A     B     C     D     E
A      0     2     4     7    -2
B     -2     0     2     5    -4
C     -4    -2     0     3    -6
D     -7    -5    -3     0    -9
E      2     4     6     9     0
row A == Bellman-Ford from A: True
C -> D goes C -> B -> E -> A -> D


### The loop order really matters

Swapping the loops so that `k` runs innermost still terminates, still fills the table,
and is still wrong - it computes "`i` to `j` via `k`" before `k`'s own row and column
are finished. Nothing crashes; a few entries are simply too large.

In [6]:
def floyd_wrong_loop_order(n, edges):
    dist = [[INF] * n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0
    for u, v, w in edges:
        dist[u][v] = min(dist[u][v], w)
    for i in range(n):
        for j in range(n):
            for k in range(n):          # k innermost - the bug
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
    return dist

bad = floyd_wrong_loop_order(N, EDGES)
diff = [(NAMES[i], NAMES[j], bad[i][j], fdist[i][j])
        for i, j in product(range(N), repeat=2) if bad[i][j] != fdist[i][j]]
for a, b, got, want in diff:
    print('%s -> %s : got %s, correct %s' % (a, b, got, want))
print('%d of %d pairs wrong, silently' % (len(diff), N * N))

A -> B : got 6, correct 2
A -> E : got 2, correct -2
E -> B : got 5, correct 4
3 of 25 pairs wrong, silently


## LeetCode 787 - Cheapest Flights Within K Stops

"At most `k` stops" means "at most `k+1` edges", and `k+1` rounds of Bellman-Ford is
precisely the set of paths with at most `k+1` edges. The constraint that looks like an
extra complication is the algorithm's own loop counter.

The trap: relax from a **copy** of the previous round. Relaxing in place lets a single
round chain several edges together, so the answer silently allows more stops than
requested.

In [7]:
def find_cheapest_price(n, flights, src, dst, k):
    dist = [INF] * n
    dist[src] = 0
    for _ in range(k + 1):
        prev = dist[:]                  # only last round's values may be extended
        for u, v, w in flights:
            if prev[u] + w < dist[v]:
                dist[v] = prev[u] + w
    return dist[dst] if dist[dst] < INF else -1

def find_cheapest_price_buggy(n, flights, src, dst, k):
    dist = [INF] * n
    dist[src] = 0
    for _ in range(k + 1):
        for u, v, w in flights:         # no copy: one round can chain 0->1->2->3
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
    return dist[dst] if dist[dst] < INF else -1

flights = [(0, 1, 100), (1, 2, 100), (2, 0, 100), (1, 3, 600), (2, 3, 200)]
for k in (1, 2):
    print('k=%d :  with copy -> %s     without copy -> %s'
          % (k, find_cheapest_price(4, flights, 0, 3, k),
                find_cheapest_price_buggy(4, flights, 0, 3, k)))

k=1 :  with copy -> 700     without copy -> 400
k=2 :  with copy -> 400     without copy -> 400


## Tests

In [8]:
assert dist == [0, 2, 4, 7, -2]
assert sdist == dist and not neg and not sneg
assert has_neg and spfa_neg and set(cyc) == {1, 2, 4}
assert fdist[0] == dist
assert fdist[2][3] == 3 and floyd_path(nxt, 2, 3) == [2, 1, 4, 0, 3]
assert len(diff) > 0
assert find_cheapest_price(4, flights, 0, 3, 1) == 700
assert find_cheapest_price(4, flights, 0, 3, 2) == 400
assert find_cheapest_price_buggy(4, flights, 0, 3, 1) == 400     # wrong on purpose
assert find_cheapest_price(3, [(0, 1, 100), (1, 2, 100)], 0, 2, 0) == -1
print('all assertions passed')

all assertions passed
